# ISSPA — Suite Completa de Presentación y Validación Cuantitativa
### Índice de Sostenibilidad Demográfico-Contributiva del Sistema Previsional Argentino

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carolinamsfelipe/Tp-Indicadores/blob/main/notebooks/ISSPA_Presentacion_Suite_Colab.ipynb)

> **Autores:** Carolina Felipe y equipo | **Cátedra:** Taller de Indicadores Económicos (UADE)
> **Finalidad:** Presentación ejecutiva y auditoría cuantitativa ante el cuerpo docente y autoridades de UADE / INECO.

---

## 1. Identidad Contable Central del ISSPA

El **ISSPA** evalúa la capacidad de soporte demográfico-contributivo intergeneracional que una cohorte nacida en el año $t$ genera a lo largo de su ciclo vital activo para sostener al stock pasivo legal que enfrentará al momento de su retiro ($t + 65$ años):

$$\text{ISSPA}_C(t) = \frac{N_t \cdot (1 - \mu) \cdot \bar{A} \cdot \tau}{J^*_{t+65} \cdot \rho}$$

* **Unidad de medida:** Años de haber medio del stock pasivo que financia la cohorte $t$.
* **Variables clave:**
  * $N_t$: Nacimientos vivos ocurridos (DEIS y RENAPER).
  * $1 - \mu$: Supervivencia acumulada de la cohorte hasta los 65 años (ONU WPP 2024).
  * $\bar{A}$: Años promedio de aporte efectivo por individuo ($14,57$ años estimados con microdatos EPH-INDEC).
  * $\tau$: Alícuota legal destinada a reparto SIPA ($21,77\%$; Ley 24.241 / Ley 27.541).
  * $J^*_{t+65}$: Stock de pasivos en edad legal de retiro (varones 65+ y mujeres 60+, Ley 24.241).
  * $\rho$: Tasa de sustitución del haber medio sobre salario medio SIPA ($40,3\%$; ANSES).

In [ ]:
# 1. Configuración y Carga de Datos (compatible con Google Colab o entorno local)
import os
import ssl
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Configuración visual profesional
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#BDC3C7'
plt.rcParams['axes.linewidth'] = 0.8

# Cargar datos: local o directo desde GitHub raw
url_github = 'https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados/tbp_modelo_avanzado_1950_2035.csv'
rutas_locales = [
    'github_staging/datos/procesados/tbp_modelo_avanzado_1950_2035.csv',
    'datos/procesados/tbp_modelo_avanzado_1950_2035.csv',
    '../datos/procesados/tbp_modelo_avanzado_1950_2035.csv'
]

df = None
for r in rutas_locales:
    if os.path.exists(r):
        df = pd.read_csv(r)
        print(f'✅ Dataset cargado desde archivo local: {r}')
        break

if df is None:
    print('🌐 Descargando dataset oficial desde GitHub para Google Colab...')
    ctx = ssl._create_unverified_context()
    req = urllib.request.Request(url_github, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, context=ctx) as resp:
        df = pd.read_csv(resp)
    print(f'✅ Dataset descargado exitosamente ({len(df)} cohortes cargadas).')

# Extracción de cohortes de referencia
c00 = df[df['cohorte'] == 2000].iloc[0]
c11 = df[df['cohorte'] == 2011].iloc[0]
c24 = df[df['cohorte'] == 2024].iloc[0]
c35 = df[df['cohorte'] == 2035].iloc[0]

isspa_2000 = c00['TBP_C_p4']
isspa_2011 = c11['TBP_C_p4']
isspa_2024 = c24['TBP_C_p4']
isspa_2035 = c35['TBP_C_p4']
drop_neto = (isspa_2024 / isspa_2000 - 1) * 100

print('='*75)
print('VALORES CENTRALES DEL ISSPA_C (CASO BASE CON CORTE LEGAL J*):')
print(f'• Cohorte 2000: {isspa_2000:.4f} años ({isspa_2000*12:.1f} meses de haber por jubilado)')
print(f'• Cohorte 2011: {isspa_2011:.4f} años ({isspa_2011*12:.1f} meses)')
print(f'• Cohorte 2024: {isspa_2024:.4f} años ({isspa_2024*12:.1f} meses)')
print(f'• Cohorte 2035: {isspa_2035:.4f} años ({isspa_2035*12:.1f} meses)')
print(f'--> Contracción neta 2000 -> 2024: {drop_neto:.2f} %')
print('='*75)

## 2. Serie Histórica y Proyección (1950–2035)
Evolución del $ISSPA_C$ a lo largo de 86 cohortes, destacando el corte legal por sexo ($J^*$) y el quiebre de natalidad de 2014.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5), dpi=120)
hist = df[(df['cohorte'] >= 1950) & (df['cohorte'] <= 2024)]
ax.plot(hist['cohorte'], hist['TBP_C_p4'], color='#1B4F72', lw=2.6, label='ISSPA_C Observado (con J* legal)')
ax.plot(hist['cohorte'], hist['TBP_C_p4_m65'], color='#7F8C8D', lw=1.5, ls='--', label='Sensibilidad: J uniforme 65+')

proj = df[(df['cohorte'] >= 2024) & (df['cohorte'] <= 2035)]
ax.plot(proj['cohorte'], proj['TBP_C_p4'], color='#B03A2E', lw=2.2, ls='-', label='Proyección 2025–2035 (Fecundidad baja 1.04)')

ax.axvline(2014, color='#E67E22', ls=':', lw=1.8, label='Quiebre de natalidad (2014)')
ax.scatter([2000, 2024], [isspa_2000, isspa_2024], color=['#1B4F72', '#B03A2E'], s=60, zorder=5)
ax.annotate(f'2000: {isspa_2000:.3f} años\n(3,7 meses)', (2000, isspa_2000), textcoords='offset points', xytext=(-20, 15),
            fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#1B4F72', alpha=0.9))
ax.annotate(f'2024: {isspa_2024:.3f} años\n(2,3 meses) [-38,9%]', (2024, isspa_2024), textcoords='offset points', xytext=(-50, -38),
            fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#B03A2E', alpha=0.9))

ax.set_title('Evolución del Soporte Demográfico-Contributivo ISSPA_C (Argentina, 1950–2035)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Cohorte de nacimiento (año t)', fontsize=10)
ax.set_ylabel('Años de haber medio por pasivo legal J*', fontsize=10)
ax.grid(True, alpha=0.3, ls='--')
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

## 3. Pilar 1: Sensibilidad Paramétrica y Gráfico Tornado
Demuestra que las elasticidades analíticas son unitarias ($|\varepsilon| = 1$) y que en ningún escenario plausible la cohorte 2024 alcanza los niveles del año 2000.

In [ ]:
escenarios_tornado = [
    ("Base (Cohorte 2024, J*)", isspa_2024, 0.0),
    ("A_bar: 50% independientes (23,6 años)", isspa_2024 * (23.58 / 14.57), (23.58 / 14.57 - 1) * 100),
    ("tau: Sector público (27%)", isspa_2024 * (0.27 / 0.2177), (0.27 / 0.2177 - 1) * 100),
    ("A_bar: Regla PP07I (17,8 años)", isspa_2024 * (17.82 / 14.57), (17.82 / 14.57 - 1) * 100),
    ("Denominador uniforme J (65+ ambos sexos)", c24['TBP_C_p4_m65'], (c24['TBP_C_p4_m65'] / isspa_2024 - 1) * 100),
    ("tau: Inciso b SIPA (23,35%)", isspa_2024 * (0.2335 / 0.2177), (0.2335 / 0.2177 - 1) * 100),
    ("tau: Efectiva SIPA neta (18,3%)", isspa_2024 * (0.183 / 0.2177), (0.183 / 0.2177 - 1) * 100),
    ("rho: Moratorias bajan al 50% (48,3%)", isspa_2024 * (0.403 / 0.483), (0.403 / 0.483 - 1) * 100),
    ("rho: Sin moratorias / contributivo puro (56,3%)", isspa_2024 * (0.403 / 0.563), (0.403 / 0.563 - 1) * 100),
]
df_t_plot = pd.DataFrame(escenarios_tornado, columns=['Escenario', 'ISSPA_C_anios', 'Var_pct'])
df_t_plot = df_t_plot[df_t_plot['Escenario'] != "Base (Cohorte 2024, J*)"].sort_values('Var_pct')

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
colores_t = ['#27AE60' if v >= 0 else '#C0392B' for v in df_t_plot['Var_pct']]
ax.barh(df_t_plot['Escenario'], df_t_plot['Var_pct'], color=colores_t, alpha=0.85, edgecolor='black', lw=0.6)
ax.axvline(0, color='black', lw=1)
ax.set_title('Gráfico Tornado: Sensibilidad del ISSPA_C ante Parámetros Alternativos (Cohorte 2024)', fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel('Variación porcentual respecto del caso base (%)', fontsize=10)
for i, v in enumerate(df_t_plot['Var_pct']):
    ax.text(v + (1.5 if v >= 0 else -6.5), i, f'{v:+.1f}%', va='center', fontsize=8.5, fontweight='bold')
ax.grid(True, alpha=0.3, ls='--', axis='x')
plt.tight_layout()
plt.show()

## 4. Pilar 2: Descomposición de Varianza de Campbell-Shiller
Demuestra algebraicamente que los nacimientos ($N_t$) explican el **107,6%** de la caída neta 2000–2024 y el **85,2%** de la varianza interanual de la serie.

In [ ]:
ln_isspa_diff = np.log(isspa_2024) - np.log(isspa_2000)
ln_N_diff = np.log(c24['N_4']) - np.log(c00['N_4'])
ln_S_diff = np.log(c24['S_tot65']) - np.log(c00['S_tot65'])
ln_J_diff = np.log(c24['J_star_4']) - np.log(c00['J_star_4'])

share_N = (ln_N_diff / ln_isspa_diff) * 100
share_J = (-ln_J_diff / ln_isspa_diff) * 100
share_S = (ln_S_diff / ln_isspa_diff) * 100

sub_var = df[(df['cohorte'] >= 2000) & (df['cohorte'] <= 2024)].copy()
d_ln_isspa = np.diff(np.log(sub_var['TBP_C_p4']))
d_ln_N = np.diff(np.log(sub_var['N_4']))
var_total = np.var(d_ln_isspa)
cov_N = np.cov(d_ln_N, d_ln_isspa)[0, 1]
share_var_N = (cov_N / var_total) * 100

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
ax.bar(['Natalidad (N)', 'Jubilados (J*)', 'Supervivencia (S65)'], [share_N, share_J, share_S],
       color=['#C0392B', '#E67E22', '#2980B9'], width=0.55, alpha=0.85, edgecolor='black', lw=0.6)
ax.axhline(0, color='black', lw=1)
ax.axhline(100, color='grey', ls=':', lw=1)
ax.set_title('Descomposición de la Caída 2000–2024: Participación de Factores (%)', fontsize=11, fontweight='bold', pad=10)
ax.set_ylabel('Participación en la caída logarítmica (%)', fontsize=10)
for i, v in enumerate([share_N, share_J, share_S]):
    ax.text(i, v + (2 if v >= 0 else -7), f'{v:+.1f}%', ha='center', fontsize=9, fontweight='bold')
ax.grid(True, alpha=0.3, ls='--', axis='y')
plt.tight_layout()
plt.show()

print(f'Natalidad explica {share_N:.1f}% de la caída neta y {share_var_N:.1f}% de la varianza interanual.')

## 5. Pilar 3: Bootstrap No Paramétrico de la EPH (1.000 Réplicas)
Cuantifica el error muestral de $\bar{A} = 14,57$ años sobre 35 ondas trimestrales de la EPH-INDEC. El intervalo al 95% es $[13,87 ; 15,30]$ años, trasladando apenas $\pm 5,1\%$ de incertidumbre al indicador.

In [ ]:
# Pilar 4: Simulación estocástica de Monte Carlo conjunta (10.000 sorteos)
np.random.seed(42)
n_mc = 10000
mc_A = np.random.triangular(left=13.5, mode=14.57, right=23.6, size=n_mc)
mc_tau = np.random.uniform(low=0.183, high=0.2335, size=n_mc)
mc_rho = np.random.triangular(left=0.371, mode=0.403, right=0.563, size=n_mc)
mc_err_demog = np.random.normal(loc=1.0, scale=0.03, size=n_mc)

isspa_mc_2024 = isspa_2024 * (mc_A / 14.57) * (mc_tau / 0.2177) * (0.403 / mc_rho) * mc_err_demog

mc_p05 = np.percentile(isspa_mc_2024, 5)
mc_p50 = np.percentile(isspa_mc_2024, 50)
mc_p95 = np.percentile(isspa_mc_2024, 95)
prob_supera = (isspa_mc_2024 >= isspa_2000).mean() * 100

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
ax.hist(isspa_mc_2024, bins=50, color='#3498DB', alpha=0.7, edgecolor='white', density=True, label='Simulación Monte Carlo (10.000 sorteos)')
ax.axvline(isspa_2024, color='#1B4F72', lw=2.2, label=f'Estimación Base 2024 ({isspa_2024:.3f} años)')
ax.axvline(mc_p50, color='#2980B9', ls='--', lw=2, label=f'Mediana P50 ({mc_p50:.3f} años)')
ax.axvline(mc_p95, color='#E67E22', ls=':', lw=2, label=f'Percentil 95 más optimista ({mc_p95:.3f} años)')
ax.axvline(isspa_2000, color='#C0392B', lw=2.5, ls='-', label=f'Cohorte 2000 de referencia ({isspa_2000:.3f} años)')

ax.set_title("Simulación de Monte Carlo del ISSPA_C (Cohorte 2024) y la 'Prueba de Oro'", fontsize=11, fontweight='bold', pad=10)
ax.set_xlabel('Años de haber medio cubiertos', fontsize=10)
ax.set_ylabel('Densidad de probabilidad', fontsize=10)
ax.grid(True, alpha=0.3, ls='--')
ax.legend(loc='upper right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
plt.show()

print(f'Prueba de Oro: P95(2024) = {mc_p95:.3f} < {isspa_2000:.3f} (2000). Prob(2024 >= 2000) = {prob_supera:.3f}% (p < 0,001).')

## 6. Pilar 5: Validación Temporal Pseudo IS vs. OOS y Test de Chow
Partición temporal In-Sample (1950–2010, N=61) frente a Out-of-Sample (2011–2024, N=14). Certifica un quiebre estructural ($F=8,40, p < 0,001$) y una brecha contrafáctica del **−40,09%** debida exclusivamente al shock de natalidad.

In [ ]:
df_eval = df[(df['cohorte'] >= 1950) & (df['cohorte'] <= 2024)].copy().reset_index(drop=True)
is_mask = (df_eval['cohorte'] >= 1950) & (df_eval['cohorte'] <= 2010)
oos_mask = (df_eval['cohorte'] >= 2011) & (df_eval['cohorte'] <= 2024)

g1 = df_eval[df_eval['cohorte'] < 2011]
g2 = df_eval[df_eval['cohorte'] >= 2011]
N = len(df_eval)

xp = np.column_stack([np.ones(N), df_eval['cohorte'].values])
yp = df_eval['TBP_C_p4'].values
bp, _, _, _ = np.linalg.lstsq(xp, yp, rcond=None)
rssp = np.sum((yp - xp @ bp) ** 2)

x1 = np.column_stack([np.ones(len(g1)), g1['cohorte'].values])
y1 = g1['TBP_C_p4'].values
b1, _, _, _ = np.linalg.lstsq(x1, y1, rcond=None)
rss1 = np.sum((y1 - x1 @ b1) ** 2)

x2 = np.column_stack([np.ones(len(g2)), g2['cohorte'].values])
y2 = g2['TBP_C_p4'].values
b2, _, _, _ = np.linalg.lstsq(x2, y2, rcond=None)
rss2 = np.sum((y2 - x2 @ b2) ** 2)

f_chow = ((rssp - (rss1 + rss2)) / 2) / ((rss1 + rss2) / (N - 4))
p_chow = 1.0 - stats.f.cdf(f_chow, 2, N - 4)

df_eval['cf_pre11'] = b1[0] + b1[1] * df_eval['cohorte']
cf_2024 = df_eval.loc[df_eval['cohorte'] == 2024, 'cf_pre11'].values[0]
gap_2024 = (isspa_2024 - cf_2024) / cf_2024 * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7.5), dpi=120, gridspec_kw={'height_ratios': [2.2, 1.2]})
ax1.plot(df_eval.loc[is_mask, 'cohorte'], df_eval.loc[is_mask, 'TBP_C_p4'], color='#1B4F72', lw=2.2, label='In-Sample (1950–2010)')
ax1.plot(df_eval.loc[oos_mask, 'cohorte'], df_eval.loc[oos_mask, 'TBP_C_p4'], color='#B03A2E', lw=2.4, label='Out-of-Sample (2011–2024)')
ax1.plot(df_eval.loc[oos_mask, 'cohorte'], df_eval.loc[oos_mask, 'cf_pre11'], color='#7F8C8D', lw=1.8, ls='--', label='Tendencia contrafáctica pre-2011')
ax1.axvline(2011, color='#E67E22', ls=':', lw=1.8, label='Corte IS / OOS (2011)')
ax1.axvline(2014, color='#C0392B', ls='-.', lw=1.5, alpha=0.8, label='Quiebre de natalidad (2014)')
ax1.fill_between(df_eval.loc[oos_mask, 'cohorte'], df_eval.loc[oos_mask, 'TBP_C_p4'], df_eval.loc[oos_mask, 'cf_pre11'], color='#F5B7B1', alpha=0.45, label='Pérdida por shock natalidad (−40%)')
ax1.set_title('Validación Temporal Pseudo IS vs. OOS: Quiebre Estructural del ISSPA_C', fontsize=11, fontweight='bold', pad=10)
ax1.set_ylabel('Años de haber medio por pasivo J*', fontsize=9.5)
ax1.grid(True, alpha=0.3, ls='--')
ax1.legend(loc='upper right', fontsize=8.5, framealpha=0.9)

oos_gaps = (df_eval.loc[oos_mask, 'TBP_C_p4'].values - df_eval.loc[oos_mask, 'cf_pre11'].values) / df_eval.loc[oos_mask, 'cf_pre11'].values * 100
ax2.bar(df_eval.loc[oos_mask, 'cohorte'], oos_gaps, color='#C0392B', width=0.6, alpha=0.85, edgecolor='black', lw=0.6)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_title('Brecha Porcentual en Out-of-Sample frente al Sendero Tendencial Histórico (%)', fontsize=10, fontweight='bold', pad=6)
ax2.set_xlabel('Cohorte de nacimiento', fontsize=9.5)
ax2.set_ylabel('Brecha (%)', fontsize=9.5)
ax2.set_ylim(-45, 5)
for x, g in zip(df_eval.loc[oos_mask, 'cohorte'], oos_gaps):
    ax2.annotate(f'{g:.1f}%', (x, g), textcoords='offset points', xytext=(0, -10), ha='center', fontsize=7.5, fontweight='bold')
ax2.grid(True, alpha=0.3, ls='--', axis='y')
plt.tight_layout()
plt.show()

print(f'Test de Chow: F = {f_chow:.2f} (p = {p_chow:.6e}). Brecha contrafáctica 2024 = {gap_2024:.2f}%.')

## 7. Tabla Síntesis Ejecutiva para la Presentación
Resumen de los cinco pilares de validación listo para defender ante el jurado.

In [ ]:
resumen = pd.DataFrame([
    {'Pilar': '1. Sensibilidad Paramétrica', 'Técnica': 'Tornado y elasticidades analíticas (|ε| = 1)', 'Resultado Cohorte 2024': '[0,137 ; 0,309] años', 'Conclusión para la Defensa': 'En ningún escenario 2024 alcanza a 2000 (0,312).'},
    {'Pilar': '2. Descomposición Campbell-Shiller', 'Técnica': 'Varianza logarítmica exacta', 'Resultado Cohorte 2024': 'N_t explica 85,2% varianza interanual', 'Conclusión para la Defensa': 'El índice está gobernado por la natalidad.'},
    {'Pilar': '3. Bootstrap EPH (A_bar)', 'Técnica': '1.000 réplicas sobre 35 ondas de EPH', 'Resultado Cohorte 2024': 'A_bar = 14,57 (IC95% [13,8 ; 15,3])', 'Conclusión para la Defensa': 'Error de encuesta traslada apenas +- 5,1% al índice.'},
    {'Pilar': '4. Simulación Monte Carlo', 'Técnica': '10.000 sorteos conjuntos', 'Resultado Cohorte 2024': 'P95 = 0,260 < 0,312 (p < 0,001)', 'Conclusión para la Defensa': 'Prueba de Oro: colapso estadísticamente indiscutible.'},
    {'Pilar': '5. Validación Temporal IS/OOS', 'Técnica': 'Test de Chow y sendero contrafáctico', 'Resultado Cohorte 2024': 'F = 8,40 (p < 0,001) | Brecha -40,1%', 'Conclusión para la Defensa': 'Quiebre estructural certificado; 40% perdido por natalidad.'}
])
from IPython.display import display, HTML
display(HTML(resumen.to_html(index=False)))
resumen.to_csv('resumen_ejecutivo_defensa_isspa.csv', index=False)
print('✅ Resumen guardado en resumen_ejecutivo_defensa_isspa.csv')

## 8. Módulo de Políticas Públicas de Natalidad: Sensibilidad en el ISSPA y Cierre de la Brecha Fiscal

Frente a la contracción del ISSPA de **0,312 a 0,191 años (2,3 meses)** y la brecha fiscal de **8,09 puntos del PIB** en 2089, este módulo simula el impacto cuantitativo de un **paquete de políticas de fomento de la natalidad y cuidado infantil**:
* **Costo Fiscal del Paquete:** ~0,20 % del PIB anual (0,15 % en jardines y salas maternales de 45 días a 3 años; licencias escalonadas con neutralidad en ANSES; asignación monotributistas ~0,01 %).
* **Canal de Transmisión:** Alivia el costo de oportunidad del tiempo parental (Teoría de Gary Becker), incentivando la fecundidad ($) y protegiendo la inserción laboral formal de la mujer ($\bar{A}$).
* **Sensibilidad:** Evalúa el desplazamiento del ISSPA ante recuperaciones de la fecundidad (de 1,04 a 1,22 y escenarios superiores) y el ahorro neto generado en puntos del PIB.

In [ ]:
# Simulación de Impacto de Políticas de Natalidad y Cuidado sobre el ISSPA y la Brecha Fiscal
isspa_base_2024 = 0.1908  # 2.3 meses de cobertura
gasto_pib_base = 10.0     # 10 % del PIB en gasto previsional consolidado histórico
brecha_base_pib = 8.09    # 8.09 puntos del PIB de asistencia de Rentas Generales en 2089

escenarios_natalidad = [
    {"escenario": "Caso Base (Sin reforma)", "tfr": 1.04, "factor_N": 1.000, "costo_paquete_pib": 0.00},
    {"escenario": "Recuperación Moderada (Licencias + Guarderías)", "tfr": 1.22, "factor_N": 1.22/1.04, "costo_paquete_pib": 0.20},
    {"escenario": "Recuperación Fuerte (Meta Nórdica)", "tfr": 1.45, "factor_N": 1.45/1.04, "costo_paquete_pib": 0.35},
    {"escenario": "Reemplazo Poblacional Puro (Techo Teórico)", "tfr": 2.10, "factor_N": 2.10/1.04, "costo_paquete_pib": 0.50}
]

resultados_sim = []
for esc in escenarios_natalidad:
    isspa_sim = isspa_base_2024 * esc["factor_N"]
    meses_cubiertos = isspa_sim * 12.0
    dias_cubiertos = isspa_sim * 365.25
    pct_cobertura = isspa_sim * 100.0
    
    asistencia_tesoro_pib = gasto_pib_base * (1.0 - isspa_sim)
    ahorro_bruto_pib = brecha_base_pib - asistencia_tesoro_pib
    ahorro_neto_pib = ahorro_bruto_pib - esc["costo_paquete_pib"]
    
    resultados_sim.append({
        "Escenario": esc["escenario"],
        "Fecundidad (Hijos/M)": esc["tfr"],
        "ISSPA (Años)": round(isspa_sim, 3),
        "Meses Cubiertos": round(meses_cubiertos, 1),
        "Días Cubiertos": round(dias_cubiertos, 0),
        "% Cobertura": f"{pct_cobertura:.1f} %",
        "Brecha Fiscal (% PIB)": round(asistencia_tesoro_pib, 2),
        "Ahorro Neto Fiscal (% PIB)": round(ahorro_neto_pib, 2)
    })

df_natalidad_sim = pd.DataFrame(resultados_sim)
print("=== SIMULACIÓN DE POLÍTICAS DE NATALIDAD Y IMPACTO EN LA SOSTENIBILIDAD PREVISIONAL ===")
print(df_natalidad_sim[["Escenario", "Fecundidad (Hijos/M)", "ISSPA (Años)", "Meses Cubiertos", "Días Cubiertos", "% Cobertura", "Brecha Fiscal (% PIB)", "Ahorro Neto Fiscal (% PIB)"]].to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
colores = ["#701A25", "#B8860B", "#2E8B57", "#002855"]
bars1 = ax1.bar(df_natalidad_sim["Fecundidad (Hijos/M)"].astype(str), df_natalidad_sim["Meses Cubiertos"], color=colores, width=0.55)
ax1.axhline(12.0, color="black", ls="--", alpha=0.5, label="1 Año Completo (Autosuficiencia)")
ax1.axhline(3.7, color="#888888", ls=":", label="Cohorte 2000 (3,7 meses)")
ax1.set_title("Meses de Jubilación Cubiertos por Cohorte según Fecundidad", fontsize=11, fontweight="bold", pad=10)
ax1.set_xlabel("Tasa Global de Fecundidad (Hijos por Mujer)", fontsize=10)
ax1.set_ylabel("Meses de Haber Financiados", fontsize=10)
ax1.set_ylim(0, 13)
for bar in bars1:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 0.3, f"{yval:.1f} m", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(True, alpha=0.2, axis="y")

bars2 = ax2.bar(df_natalidad_sim["Fecundidad (Hijos/M)"].astype(str), df_natalidad_sim["Brecha Fiscal (% PIB)"], color=colores, width=0.55)
ax2.set_title("Asistencia Requerida de Rentas Generales en 2089 (Puntos del PIB)", fontsize=11, fontweight="bold", pad=10)
ax2.set_xlabel("Tasa Global de Fecundidad (Hijos por Mujer)", fontsize=10)
ax2.set_ylabel("Puntos del PIB del Tesoro", fontsize=10)
ax2.set_ylim(0, 10)
for bar in bars2:
    yval = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, yval + 0.2, f"{yval:.2f} %", ha="center", va="bottom", fontsize=9, fontweight="bold")
ax2.grid(True, alpha=0.2, axis="y")

plt.tight_layout()


# ----------------------------------------------------------------------------------
# MATRIZ MACROECONÓMICA INTEGRADA: Crecimiento Real (g) x Formalización (A) x Natalidad (Nt)
# ----------------------------------------------------------------------------------
escenarios_macro = [
    {"Escenario": "1. Estanflación Inercial (Sin reformas)", "g": 0.005, "A": 14.57, "tfr": 1.04, "pib_base": 600000.0},
    {"Escenario": "2. Reformas Moderadas (Caso Base)", "g": 0.015, "A": 18.00, "tfr": 1.22, "pib_base": 600000.0},
    {"Escenario": "3. Shock Productividad y Capital Humano", "g": 0.025, "A": 22.00, "tfr": 1.35, "pib_base": 600000.0}
]

res_macro = []
for m in escenarios_macro:
    pib_2089 = m["pib_base"] * ((1.0 + m["g"]) ** 65)
    f_N = m["tfr"] / 1.04
    f_A = m["A"] / 14.57
    isspa_macro = min(0.1908 * f_N * f_A, 1.0)
    meses_m = isspa_macro * 12.0
    dias_m = isspa_macro * 365.25
    brecha_pib_m = 10.0 * (1.0 - isspa_macro)
    brecha_usd_m = (brecha_pib_m / 100.0) * pib_2089
    
    res_macro.append({
        "Escenario Macroeconómico": m["Escenario"],
        "Tasa g real": f"{m['g']*100:.1f} %",
        "PIB 2089 (USD M)": f"USD {pib_2089:,.0f} M",
        "Densidad A": f"{m['A']:.1f} años",
        "Fecundidad": f"{m['tfr']:.2f}",
        "ISSPA_C (Años)": round(isspa_macro, 3),
        "Cobertura": f"{meses_m:.1f} m ({dias_m:.0f} d)",
        "Brecha (% PIB)": f"{brecha_pib_m:.2f} % PIB",
        "Brecha (USD M)": f"USD {brecha_usd_m:,.0f} M"
    })

df_macro_out = pd.DataFrame(res_macro)
print("")
print("=== MATRIZ MACROECONÓMICA INTEGRADA (2024-2089): CRECIMIENTO g, EMPLEO A Y NATALIDAD ===")
print(df_macro_out[["Escenario Macroeconómico", "Tasa g real", "PIB 2089 (USD M)", "Densidad A", "Fecundidad", "ISSPA_C (Años)", "Cobertura", "Brecha (% PIB)"]].to_string(index=False))

plt.show()


## 9. Estimación de Natalidad en un Quinquenio (2025–2030): Benchmark Cuasi-Experimental de Alemania

Para modelar el sendero de fecundidad sin supuestos arbitrarios, este módulo traslada la elasticidad empírica de la reforma alemana (*Elterngeld* + ley de guarderías *KiföG*, Raute 2019; Bauernschuster & Schlotter 2015), donde la fecundidad subió de **1,39 a 1,60 en 5 años (+15,1 %)**.
* **Trayectoria 5 años en Argentina:** Fecundidad crece de **1,04 a 1,22 hijos/mujer** (+17,3 % acumulado), incorporando **+88.600 nacimientos anuales** hacia 2030.
* **Impacto en el ISSPA:** La cohorte 2030 gana **12 días de jubilación cubierta** (de 70 a 82 días).
* **Retorno Fiscal Neto:** Genera un ahorro de **0,33 puntos del PIB** en 2089 frente a un costo de intervención de 0,20 % del PIB.

In [ ]:
# Estimación de Natalidad en un Quinquenio (2025-2030) - Benchmark Alemania
anios_quinq = np.arange(2025, 2031)
tfr_quinq = np.array([1.04, 1.07, 1.11, 1.15, 1.18, 1.22])
nacimientos_base = 512000.0
nacimientos_quinq = nacimientos_base * (tfr_quinq / 1.04)

isspa_quinq = 0.1908 * (tfr_quinq / 1.04)
dias_quinq = isspa_quinq * 365.25
brecha_quinq = 10.0 * (1.0 - isspa_quinq)

df_quinq = pd.DataFrame({
    "Año": anios_quinq,
    "TFR (Hijos/M)": tfr_quinq,
    "Nacimientos Proyectados": [f"{int(round(n)):,}" for n in nacimientos_quinq],
    "Crecimiento Acumulado (%)": [f"{((t/1.04)-1)*100:+.1f} %" for t in tfr_quinq],
    "ISSPA_C (Años)": np.round(isspa_quinq, 3),
    "Días Cubiertos": np.round(dias_quinq, 0).astype(int),
    "Brecha Fiscal (% PIB)": np.round(brecha_quinq, 2)
})

print("=== TRAYECTORIA QUINQUENAL DE FECUNDIDAD Y SOSTENIBILIDAD PREVISIONAL (2025-2030) ===")
print(df_quinq.to_string(index=False))

fig, ax_tfr = plt.subplots(figsize=(10, 4.8), dpi=120)
ax_tfr.plot(anios_quinq, tfr_quinq, color="#002855", marker="o", lw=2.5, markersize=8, label="TFR Proyectada (Alemania Benchmark)")
ax_tfr.axhline(1.04, color="#701A25", ls="--", alpha=0.6, label="Línea Base 2024 (1,04 hijos/mujer)")
for a, t, d in zip(anios_quinq, tfr_quinq, dias_quinq):
    txt = f"{t:.2f} ({int(round(d))} d)"
    ax_tfr.annotate(txt, (a, t), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=8.5, fontweight="bold")
ax_tfr.set_title("Sendero de Recuperación de Fecundidad e Impacto en Días Cubiertos del ISSPA (2025-2030)", fontsize=11, fontweight="bold", pad=12)
ax_tfr.set_xlabel("Año de Implementación de Políticas", fontsize=10)
ax_tfr.set_ylabel("Tasa Global de Fecundidad (Hijos por Mujer)", fontsize=10)
ax_tfr.set_ylim(1.00, 1.30)
ax_tfr.grid(True, alpha=0.25)
ax_tfr.legend(loc="upper left", frameon=True)
plt.tight_layout()
plt.show()


## 10. Módulo de Reforma Estratégica: Fondo Soberano de Vaca Muerta (2026–2089)

Simulación cuantitativa y actuarial del **Fondo Soberano de Vaca Muerta (FSVM)** bajo tres escenarios de aporte (Constante, Caso Base con pico en 2034, y Pesimista).
* **Ahorro puro vs. Regla del 3%**: Permite evaluar la acumulación de capital frente al retiro anual del 3% desde 2050 (que cubre permanentemente entre 4% y 5% de las jubilaciones argentinas sin consumir el principal).
* **Sensibilidad de Cartera (Asset Allocation)**: Cruza los tres senderos de exportaciones con tres estrategias de inversión internacional: Conservador (2,0% real), Balanceado (3,5% real) y Agresivo tipo Noruega (4,5% real, capaz de financiar el 63% de todas las jubilaciones de 2089).


In [ ]:
# =============================================================================
# 10. Módulo de Reforma Estratégica: Fondo Soberano de Vaca Muerta (2026–2089)
#
#    Tres escenarios de aportes al fondo:
#    A) Constante  : USD 3.000 M todos los años (Cuadro 6 original).
#    B) Base       : aporte = 12 % de exportaciones energéticas. Suben hasta
#                    un pico en 2034 y después bajan gradualmente, sin llegar
#                    nunca a cero (cola de producción madura hasta 2088).
#    C) Pesimista  : pico más bajo y corto (solo unos pocos años > USD 3.000 M),
#                    declinación temprana hacia una cola de aportes chicos.
#
#    En los tres escenarios hay aportes todos los años entre 2026 y 2088.
#
#    Además, cada escenario se simula de dos formas:
#    - Ahorro puro: no se retira nada hasta 2089 (línea llena).
#    - Regla del 3 %: desde 2050 se retira cada año el 3 % del capital para
#      pagar jubilaciones durante el estrangulamiento (línea punteada).
#      Como el fondo rinde 3,5 %, el capital no se consume: sigue creciendo.
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- PARÁMETROS GENERALES (cambiar acá) ----------------
r_real      = 0.035      # rendimiento real anual del fondo
g_pib       = 0.015      # crecimiento real del PIB
pib_2089    = 1_534_000  # PIB 2089 en USD M (constantes)
brecha_pib  = 8.09       # descalce previsional 2089 (% del PIB), caso base
gasto_prev  = 10.0       # gasto previsional total (% del PIB)
share_expo  = 0.12       # % de exportaciones energéticas que va al fondo
anios_anual = 15         # años de la anualidad de desacumulación
regla_inicio = 2050      # desde este año el fondo empieza a pagar jubilaciones
regla_tasa   = 0.03      # retiro anual = 3 % del capital (regla fiscal noruega)

# Perfiles de inversión del fondo (rendimiento real anual) para la sensibilidad
perfiles = {
    'Conservador (70 % bonos AAA / TIPS)':        0.020,
    'Balanceado (60 % acciones / 40 % bonos)':     0.035,
    'Agresivo tipo Noruega (~70 % acciones)':      0.045,  # Noruega: ~4,2–4,5 % real neto desde 1998
}

# ---------------- TRAYECTORIAS DE EXPORTACIONES (USD M) ----------------
# Puntos de anclaje {año: exportaciones}. Entre puntos se interpola linealmente,
# así el flujo cambia todos los años. Valores ILUSTRATIVOS: reemplazar por
# proyecciones con fuente (Secretaría de Energía, Rystad, CEPH, etc.).
expo_B = {2026: 15_000, 2030: 32_000, 2034: 40_000, 2038: 37_000,
          2045: 30_000, 2055: 18_000, 2065: 11_000, 2088: 6_000}
expo_C = {2026: 15_000, 2029: 26_500, 2031: 28_000, 2034: 23_000,
          2040: 14_000, 2050: 7_000, 2060: 4_500, 2088: 2_500}

anios = np.arange(2026, 2090)  # 2026 ... 2089

def trayectoria(anclas):
    x = np.array(list(anclas.keys())); y = np.array(list(anclas.values()), dtype=float)
    return np.interp(anios, x, y)

def simular(aportes, r=None):
    """capital[y] = capital al INICIO del año y (antes del aporte de y).
    Aportes 2026–2088 -> 63 aportes, consistente con el Cuadro 6."""
    r = r_real if r is None else r
    cap = np.zeros(len(anios))
    for i in range(1, len(anios)):
        cap[i] = cap[i - 1] * (1 + r) + aportes[i - 1]
    return cap

def pib(y):
    return pib_2089 / (1 + g_pib) ** (2089 - y)

ultimo = anios <= 2088
escenarios = {
    'A: Constante USD 3.000 M': np.where(ultimo, 3_000.0, 0.0),
    'B: Base (pico 2034)':      np.where(ultimo, share_expo * trayectoria(expo_B), 0.0),
    'C: Pesimista (pico corto)': np.where(ultimo, share_expo * trayectoria(expo_C), 0.0),
}
colores = {'A: Constante USD 3.000 M': '#7F8C8D',
           'B: Base (pico 2034)': '#002855',
           'C: Pesimista (pico corto)': '#B03A2E'}
capitales = {k: simular(v) for k, v in escenarios.items()}

def simular_con_regla(aportes):
    """Igual que simular(), pero desde regla_inicio se retira cada año el
    regla_tasa del capital al inicio del año. Devuelve capital y retiros."""
    cap = np.zeros(len(anios)); ret = np.zeros(len(anios))
    for i in range(1, len(anios)):
        k = cap[i - 1]
        if anios[i - 1] >= regla_inicio:
            ret[i - 1] = regla_tasa * k
            k -= ret[i - 1]
        cap[i] = k * (1 + r_real) + aportes[i - 1]
    if anios[-1] >= regla_inicio:
        ret[-1] = regla_tasa * cap[-1]
    return cap, ret

con_regla = {k: simular_con_regla(v) for k, v in escenarios.items()}
gasto_usd = np.array([gasto_prev / 100 * pib(y) for y in anios])   # gasto previsional en USD M

# ---------------- CUADRO DE HITOS ----------------
hitos = [2035, 2050, 2065, 2089]
filas = []
for nombre, cap in capitales.items():
    for y in hitos:
        k = cap[anios == y][0]
        filas.append({'Escenario': nombre, 'Año': y,
                      'Capital (USD M)': round(k),
                      '% del PIB': round(k / pib(y) * 100, 1),
                      'Renta anual real (% PIB)': round(k * r_real / pib(y) * 100, 2)})
cuadro = pd.DataFrame(filas)

# ---------------- COBERTURA EN 2089 ----------------
fac_anual = r_real / (1 - (1 + r_real) ** (-anios_anual))
cob = []
for nombre, cap in capitales.items():
    ap = escenarios[nombre]; k = cap[-1]
    perp = k * r_real / pib_2089 * 100
    anual = k * fac_anual / pib_2089 * 100
    sobre = anios[ap > 3_000]
    cob.append({'Escenario': nombre,
                'Total aportado (USD M)': round(ap.sum()),
                'Aporte máximo (USD M)': round(ap.max()),
                'Años con aporte > USD 3.000 M': len(sobre),
                'Aporte en 2088 (USD M)': round(ap[anios == 2088][0]),
                'Capital 2089 (USD M)': round(k),
                'Capital 2089 (% PIB)': round(k / pib_2089 * 100, 1),
                'Renta perpetua (% PIB)': round(perp, 2),
                f'Anualidad {anios_anual} años (% PIB)': round(anual, 2),
                'Cubre brecha - perpetua (%)': round(perp / brecha_pib * 100, 1),
                'Cubre brecha - anualidad (%)': round(anual / brecha_pib * 100, 1),
                'Cubre gasto previsional - perpetua (%)': round(perp / gasto_prev * 100, 1),
                'Años de brecha que paga el capital': round(k / (brecha_pib / 100 * pib_2089), 1)})
cobertura = pd.DataFrame(cob).set_index('Escenario').T

try:
    from IPython.display import display
    display(cuadro); display(cobertura)
except ImportError:
    print(cuadro.to_string(index=False)); print(cobertura.to_string())


# ---------------- REGLA DEL 3 %: CUÁNTO PAGA DEL GASTO PREVISIONAL ----------------
filas_r = []
for nombre, (cap_r, ret) in con_regla.items():
    fila = {'Escenario': nombre}
    for y in [2050, 2065, 2089]:
        i = np.where(anios == y)[0][0]
        fila[f'Retiro {y} (USD M)'] = round(ret[i])
        fila[f'Cubre gasto {y} (%)'] = round(ret[i] / gasto_usd[i] * 100, 1)
    fila['Total retirado 2050–2089 (USD M)'] = round(ret.sum())
    fila['Capital 2089 con regla (USD M)'] = round(cap_r[-1])
    fila['Capital 2089 con regla (% PIB)'] = round(cap_r[-1] / pib_2089 * 100, 1)
    filas_r.append(fila)
regla = pd.DataFrame(filas_r).set_index('Escenario').T
try:
    display(regla)
except NameError:
    print(regla.to_string())

# ---------------- GRÁFICO ----------------
# Momentos demográficos del ISSPA (sección 5.1 del informe)
fases = [
    (2026, 2049.5, '#27AE60', 'Bono demográfico\ncohortes 1985–2000 en edad activa\n→ AHORRAR'),
    (2050, 2065,   '#C0392B', 'Estrangulamiento\nse jubilan las cohortes 1985–2000\n→ USAR EL FONDO'),
]
hitos_v = [(2033, 'Cohorte 2014 entra\nal mercado laboral'),
           (2089, 'Retiro cohorte 2024\n(aportes cubren 19,1 %)')]

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 11), dpi=120,
                                    gridspec_kw={'height_ratios': [2.4, 1.1, 1.1]}, sharex=True)
for ax in (ax1, ax2, ax3):
    for a, b, col, _ in fases:
        ax.axvspan(a, b, color=col, alpha=0.07, lw=0)
    for x, _ in hitos_v:
        ax.axvline(x, color='#555555', ls=':', lw=1.2)
    ax.grid(True, alpha=0.25)

# Panel 1: capital
for nombre, cap in capitales.items():
    ls = '--' if nombre.startswith('A') else '-'
    ax1.plot(anios, cap / 1000, color=colores[nombre], lw=2.4, ls=ls, label=f'{nombre} · ahorro puro')
    v = cap[-1]
    ax1.annotate(f'{v/1000:,.0f} mil M ({v/pib_2089*100:.1f} % PIB)', (2089, v / 1000),
                 textcoords='offset points', xytext=(6, {'A': 6, 'B': -12, 'C': -4}[nombre[0]]),
                 fontsize=8.5, fontweight='bold', color=colores[nombre])
for nombre, (cap_r, _) in con_regla.items():
    ax1.plot(anios, cap_r / 1000, color=colores[nombre], lw=1.6, ls=':',
             label=f'{nombre[:1]} · con retiro del {regla_tasa*100:.0f} % desde {regla_inicio}')
ymax = max(c.max() for c in capitales.values()) / 1000 * 1.15
ax1.set_ylim(0, ymax)
for a, b, col, txt in fases:
    ax1.text((a + b) / 2, ymax * 0.97, txt, ha='center', va='top', fontsize=8.5,
             fontweight='bold', color=col)
ax1.text(2033.6, ymax * 0.40, hitos_v[0][1], ha='left', va='top', fontsize=8, color='#555555')
ax1.text(2088.4, ymax * 0.62, hitos_v[1][1], ha='right', va='top', fontsize=8, color='#555555')
ax1.set_xlim(2025, 2101)
ax1.set_ylabel('Capital acumulado\n(miles de M USD)')
ax1.set_title('Fondo Soberano de Vaca Muerta: ahorrar en el bono demográfico, usar en el estrangulamiento',
              fontsize=12, fontweight='bold')
ax1.legend(loc='upper left', bbox_to_anchor=(0.37, 0.84), fontsize=7.8, ncol=1, framealpha=0.9)

# Panel 2: aportes
for nombre, ap in escenarios.items():
    ls = '--' if nombre.startswith('A') else '-'
    ax2.plot(anios[ultimo], ap[ultimo], color=colores[nombre], lw=2.2, ls=ls, label=nombre)
ax2.fill_between(anios[ultimo], escenarios['B: Base (pico 2034)'][ultimo], color=colores['B: Base (pico 2034)'], alpha=0.10)
ax2.fill_between(anios[ultimo], escenarios['C: Pesimista (pico corto)'][ultimo], color=colores['C: Pesimista (pico corto)'], alpha=0.12)
ax2.set_ylabel('Aporte anual\n(USD M)')
ax2.legend(loc='upper right', fontsize=8)

# Panel 3: retiros con regla del 3 % como % del gasto previsional
for nombre, (_, ret) in con_regla.items():
    m = anios >= regla_inicio
    ls = '--' if nombre.startswith('A') else '-'
    ax3.plot(anios[m], ret[m] / gasto_usd[m] * 100, color=colores[nombre], lw=2.2, ls=ls, label=nombre)
ax3.set_ylim(bottom=0)
ax3.set_ylabel(f'Retiro del {regla_tasa*100:.0f} %\n(% del gasto previsional)')
ax3.set_xlabel('Año')
ax3.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

# =============================================================================
# SENSIBILIDAD AL PERFIL DE INVERSIÓN: 3 escenarios de aportes x 3 rendimientos
# =============================================================================
filas_s = []
for nombre, ap in escenarios.items():
    for perfil, r in perfiles.items():
        k = simular(ap, r)[-1]
        fac = r / (1 - (1 + r) ** (-anios_anual))
        anual = k * fac / pib_2089 * 100
        filas_s.append({'Aportes': nombre, 'Perfil': perfil, 'Rend. real': f'{r*100:.1f} %',
                        'Capital 2089 (USD M)': round(k),
                        'Capital 2089 (% PIB)': round(k / pib_2089 * 100, 1),
                        'Anualidad 15 años (% PIB)': round(anual, 2),
                        'Cubre brecha (%)': round(anual / brecha_pib * 100, 1),
                        'Cubre gasto previsional (%)': round(anual / gasto_prev * 100, 1)})
sensibilidad = pd.DataFrame(filas_s)
try:
    display(sensibilidad)
except NameError:
    print(sensibilidad.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5.5), dpi=120)
nombres_cortos = {'A: Constante USD 3.000 M': 'A: Constante\n(USD 3.000 M fijos)',
                  'B: Base (pico 2034)': 'B: Base\n(pico 2034)',
                  'C: Pesimista (pico corto)': 'C: Pesimista\n(pico corto)'}
colores_perfil = ['#A9CCE3', '#2E86C1', '#1B2631']
x = np.arange(len(escenarios)); ancho = 0.26
for j, (perfil, r) in enumerate(perfiles.items()):
    vals = sensibilidad[sensibilidad['Perfil'] == perfil]['Cubre gasto previsional (%)'].values
    barras = ax.bar(x + (j - 1) * ancho, vals, ancho, color=colores_perfil[j],
                    label=f'{perfil.split(" (")[0]} · rinde {r*100:.1f} % real')
    for b, v in zip(barras, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 1, f'{v:.0f} %', ha='center',
                fontsize=9.5, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels([nombres_cortos[n] for n in escenarios], fontsize=10)
ax.set_ylabel('% de las jubilaciones de 2089\nque paga el fondo')
ax.set_ylim(0, sensibilidad['Cubre gasto previsional (%)'].max() * 1.18)
ax.set_title('¿Cuánto de las jubilaciones de 2089 paga el fondo?\n'
             'Según cuánto aporte Vaca Muerta (grupos) y cómo se invierta (colores)',
             fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.25, axis='y')
ax.legend(loc='upper right', fontsize=9, title='Cómo se invierte el fondo', title_fontsize=9)
plt.tight_layout(); plt.show()

